In [1]:
#@title 0) Config (edit me)

import os
from pathlib import Path

# --------------------
# Data locations
# --------------------
LENS_COMPRESSION_DRIVE_ROOT = "/content/drive/MyDrive/jianglens"  # base folder on Drive
LENS_COMPRESSION_YOUTUBE_ROOT = LENS_COMPRESSION_DRIVE_ROOT + "/youtube"  # contains <channel_id>/<video_id>/audio.wav
LENS_COMPRESSION_WORK_ROOT = "/content/jianglens"  # fast local workspace

# --------------------
# Environment cache (stored on Drive)
# --------------------
ENV_CACHE_ROOT = "/content/drive/MyDrive/jianglens/_colab_envs"
ENV_NAME = "pyannote4"
ENV_VERSION = "v1"  # bump this to force a new cached env; set "" to auto-hash from spec

# --------------------
# HuggingFace model cache (stored on Drive)
# --------------------
HF_HOME = LENS_COMPRESSION_DRIVE_ROOT + "/_hf_home"  # persists model downloads across sessions

# --------------------
# Micromamba env spec
# --------------------
MAMBA_ROOT = "/content/micromamba"  # local path (fast). restored from Drive cache when available.
MAMBA_ENV_NAME = "audio"
PYTHON_VERSION = "3.10"
CONDA_PACKAGES = [
    "libsndfile",
    "ffmpeg",
]

# Pip steps run inside the micromamba env's python.
# Keep these as simple lists so you can reuse this notebook for other env presets.
PIP_STEPS = [
    ["install", "pyannote.audio==4.0.3"],
    ["uninstall", "torchcodec", "-y"],
    [
        "install",
        "torch",
        "torchvision",
        "torchaudio",
        "--index-url",
        "https://download.pytorch.org/whl/cu121",
        "--force-reinstall",
        "numpy<2",
    ],
]

FORCE_REBUILD_ENV = False  # set True to ignore cache and rebuild

# --------------------
# Worker tuning / batch
# --------------------
MAX_WORKERS = 2  # L4/T4: 2 is usually safe; 3+ may OOM
RUN_BATCH = True  # set True to start processing when you run the batch cell

DIARIZATION_PIPELINE_ID = "pyannote/speaker-diarization-3.1"
SEGMENTATION_BATCH_SIZE=128
EMBEDDING_BATCH_SIZE=128
GROUP_THRESHOLD_SECONDS = 3.0

# Optional: quick sanity check on one video (channel_id/video_id)
TEST_VIDEO = "@PredictiveHistory/0HYET47Cc-E"  # e.g. "UCxxxxxxxx/s_8BKJDiwKw"

# --------------------
# Mount Drive + load HF token
# --------------------

def _is_colab() -> bool:
    try:
        import google.colab  # type: ignore

        return True
    except Exception:
        return False


HF_TOKEN = None

if _is_colab():
    from google.colab import drive, userdata  # type: ignore

    drive.mount("/content/drive")

    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

HF_TOKEN = (
    HF_TOKEN
    or os.environ.get("HF_TOKEN")
    or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    or os.environ.get("HUGGINGFACE_TOKEN")
)

if not HF_TOKEN:
    HF_TOKEN = input("Enter HuggingFace token (HF_TOKEN): ").strip()

Path(LENS_COMPRESSION_WORK_ROOT).mkdir(parents=True, exist_ok=True)
Path(ENV_CACHE_ROOT).mkdir(parents=True, exist_ok=True)
Path(HF_HOME).mkdir(parents=True, exist_ok=True)

# Point huggingface_hub at the Drive-backed cache so models persist across sessions
os.environ["HF_HOME"] = HF_HOME

print("✅ Config loaded")
print(f"   HF_HOME → {HF_HOME}")
print(f"   YouTube root → {LENS_COMPRESSION_YOUTUBE_ROOT}")

Mounted at /content/drive
✅ Config loaded
   HF_HOME → /content/drive/MyDrive/jianglens/_hf_home
   YouTube root → /content/drive/MyDrive/jianglens/youtube


In [2]:
#@title 1) Environment setup (cached)
#@markdown Restores a cached micromamba env from Drive if available; otherwise builds + caches it.

from __future__ import annotations

import hashlib
import json
import shutil
import subprocess
import time
from dataclasses import asdict, dataclass
from pathlib import Path


@dataclass(frozen=True)
class MambaEnvSpec:
    name: str
    version: str | None
    env_name: str
    python: str
    conda_packages: tuple[str, ...]
    conda_channels: tuple[str, ...]
    pip_steps: tuple[tuple[str, ...], ...]

    def fingerprint(self) -> str:
        payload = json.dumps(asdict(self), sort_keys=True, separators=(",", ":")).encode(
            "utf-8"
        )
        return hashlib.sha256(payload).hexdigest()[:10]

    def cache_id(self) -> str:
        parts = [self.name]
        if self.version:
            parts.append(self.version)
        parts.append(self.fingerprint())
        return "__".join(parts)


class MicromambaEnvCache:
    def __init__(self, *, spec: MambaEnvSpec, mamba_root: Path, cache_root: Path):
        self.spec = spec
        self.mamba_root = mamba_root
        self.cache_root = cache_root

    @property
    def cache_tar_path(self) -> Path:
        return self.cache_root / f"{self.spec.cache_id()}.tar.gz"

    @property
    def cache_manifest_path(self) -> Path:
        return self.cache_root / f"{self.spec.cache_id()}.json"

    @property
    def python_bin(self) -> Path:
        return self.mamba_root / "envs" / self.spec.env_name / "bin" / "python"

    @property
    def pip_bin(self) -> Path:
        return self.mamba_root / "envs" / self.spec.env_name / "bin" / "pip"

    @property
    def lib_dir(self) -> Path:
        return self.mamba_root / "envs" / self.spec.env_name / "lib"

    def _run(self, args: list[str]) -> None:
        print("$", " ".join(args))
        subprocess.run(args, check=True)

    def _ensure_micromamba_bin(self) -> Path:
        bin_path = Path("/content/bin/micromamba")
        if bin_path.exists():
            return bin_path

        Path("/content/bin").mkdir(parents=True, exist_ok=True)
        cmd = "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba"
        print("$", cmd)
        subprocess.run(cmd, shell=True, check=True)

        if not bin_path.exists():
            raise FileNotFoundError(f"micromamba not found at {bin_path}")
        return bin_path

    def restore(self) -> None:
        if not self.cache_tar_path.exists():
            raise FileNotFoundError(f"Cache not found: {self.cache_tar_path}")

        if self.mamba_root.exists():
            shutil.rmtree(self.mamba_root)

        tmp_tar = Path("/tmp/env.tar.gz")
        shutil.copyfile(self.cache_tar_path, tmp_tar)

        print(f"♻️ Restoring env from cache: {self.cache_tar_path}")
        self._run(["tar", "-xzf", str(tmp_tar), "-C", str(self.mamba_root.parent)])

        tmp_tar.unlink(missing_ok=True)

        if not self.python_bin.exists():
            raise RuntimeError(f"Restore succeeded but python missing: {self.python_bin}")

    def install(self) -> None:
        micromamba = self._ensure_micromamba_bin()

        if self.mamba_root.exists():
            shutil.rmtree(self.mamba_root)

        create_cmd = [
            str(micromamba),
            "create",
            "-r",
            str(self.mamba_root),
            "-n",
            self.spec.env_name,
            "-y",
            f"python={self.spec.python}",
            *self.spec.conda_packages,
        ]
        for channel in self.spec.conda_channels:
            create_cmd.extend(["-c", channel])

        print("📦 Creating micromamba env…")
        self._run(create_cmd)

        if not self.pip_bin.exists():
            raise RuntimeError(f"pip missing: {self.pip_bin}")

        print("🐍 Running pip steps…")
        for step in self.spec.pip_steps:
            self._run([str(self.pip_bin), *step])

        print("✅ Environment ready")

    def backup(self) -> None:
        self.cache_root.mkdir(parents=True, exist_ok=True)

        if not self.mamba_root.exists():
            raise FileNotFoundError(f"Nothing to backup at {self.mamba_root}")

        tmp_tar = Path("/tmp") / f"{self.spec.cache_id()}.tar.gz"
        tmp_manifest = Path("/tmp") / f"{self.spec.cache_id()}.json"

        print(f"📦 Creating cache tar: {tmp_tar}")
        self._run(
            [
                "tar",
                "-czf",
                str(tmp_tar),
                "-C",
                str(self.mamba_root.parent),
                self.mamba_root.name,
            ]
        )

        manifest = {
            "created": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "spec": asdict(self.spec),
        }
        tmp_manifest.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

        shutil.copyfile(tmp_tar, self.cache_tar_path)
        shutil.copyfile(tmp_manifest, self.cache_manifest_path)

        tmp_tar.unlink(missing_ok=True)
        tmp_manifest.unlink(missing_ok=True)

        print(f"✅ Cached env saved to: {self.cache_tar_path}")

    def ensure(self, *, force_rebuild: bool = False) -> None:
        if not force_rebuild and self.cache_tar_path.exists():
            self.restore()
            return

        print("🛠️ Cache miss (or forced rebuild). Building environment…")
        self.install()
        self.backup()


spec = MambaEnvSpec(
    name=ENV_NAME,
    version=ENV_VERSION.strip() or None,
    env_name=MAMBA_ENV_NAME,
    python=PYTHON_VERSION,
    conda_packages=tuple(CONDA_PACKAGES),
    conda_channels=("conda-forge",),
    pip_steps=tuple(tuple(step) for step in PIP_STEPS),
)

ENV = MicromambaEnvCache(
    spec=spec,
    mamba_root=Path(MAMBA_ROOT),
    cache_root=Path(ENV_CACHE_ROOT),
)
ENV.ensure(force_rebuild=FORCE_REBUILD_ENV)

PYTHON_BIN = str(ENV.python_bin)
LD_LIB_DIR = str(ENV.lib_dir)

print("PYTHON_BIN:", PYTHON_BIN)


♻️ Restoring env from cache: /content/drive/MyDrive/jianglens/_colab_envs/pyannote4__v1__cdf3f26c21.tar.gz
$ tar -xzf /tmp/env.tar.gz -C /content
PYTHON_BIN: /content/micromamba/envs/audio/bin/python


In [3]:
#@title 2) Write worker (implementation)
#@markdown Worker code


from pathlib import Path

worker_script = r"""
import os
import json
import shutil
import matplotlib

matplotlib.use("Agg")

import torch
import torchaudio
from pathlib import Path
from pyannote.audio import Pipeline
from pyannote.core import Annotation, Segment

# --- PATCHES (torch/pyannote compatibility) ---
try:
    from pyannote.audio.core.task import Specifications

    torch.serialization.add_safe_globals([Specifications])
except Exception:
    pass

if not hasattr(torch, "_original_load_safe"):
    torch._original_load_safe = torch.load


def _unrestricted_load(*args, **kwargs):
    if "weights_only" not in kwargs:
        kwargs["weights_only"] = False
    return torch._original_load_safe(*args, **kwargs)


torch.load = _unrestricted_load
# ---------------------------------------------


LENS_COMPRESSION_DRIVE_ROOT = os.environ.get("LENS_COMPRESSION_DRIVE_ROOT", "/content/drive/MyDrive/jianglens")
LENS_COMPRESSION_YOUTUBE_ROOT = os.environ.get("LENS_COMPRESSION_YOUTUBE_ROOT") or (LENS_COMPRESSION_DRIVE_ROOT + "/youtube")
LENS_COMPRESSION_WORK_ROOT = os.environ.get("LENS_COMPRESSION_WORK_ROOT", "/content/jianglens")

PIPELINE_ID = os.environ.get("DIARIZATION_PIPELINE_ID", "pyannote/speaker-diarization-3.1")
SEGMENTATION_BATCH_SIZE = int(os.environ.get("SEGMENTATION_BATCH_SIZE", "8"))
EMBEDDING_BATCH_SIZE = int(os.environ.get("EMBEDDING_BATCH_SIZE", "8"))
GROUP_THRESHOLD_SECONDS = float(os.environ.get("GROUP_THRESHOLD_SECONDS", "3.0"))


def serialize(diarization, embeddings=None):
    if not hasattr(diarization, "itertracks"):
        if hasattr(diarization, "annotation"):
            diarization = diarization.annotation
        elif hasattr(diarization, "speaker_diarization"):
            diarization = diarization.speaker_diarization

    rows = []
    for segment, track, label in diarization.itertracks(yield_label=True):
        rows.append(
            {
                "speaker": label,
                "start": round(segment.start, 3),
                "end": round(segment.end, 3),
                "track": track,
            }
        )

    embeddings_dict = embeddings if embeddings is not None else {}
    return {"diarization": rows, "embeddings": embeddings_dict}


def group(diarization, threshold=3.0):
    if not hasattr(diarization, "itertracks"):
        if hasattr(diarization, "annotation"):
            diarization = diarization.annotation
        elif hasattr(diarization, "speaker_diarization"):
            diarization = diarization.speaker_diarization

    grouped = Annotation()
    curr_spk, curr_seg = None, None

    for seg, track, label in diarization.itertracks(yield_label=True):
        if curr_spk is None:
            curr_spk, curr_seg = label, seg
            continue

        if curr_spk == label and (seg.start - curr_seg.end <= threshold):
            curr_seg = Segment(curr_seg.start, seg.end)
        else:
            grouped[curr_seg] = curr_spk
            curr_spk, curr_seg = label, seg

    if curr_spk:
        grouped[curr_seg] = curr_spk

    return grouped


def main(video_path: str) -> int:
    token = os.environ.get("HF_TOKEN")
    if not token:
        raise ValueError("HF_TOKEN is missing. Pass it as an env var.")

    print(f"🎬 [Worker] Processing: {video_path}")

    drive_dir = Path(LENS_COMPRESSION_YOUTUBE_ROOT) / video_path
    work_dir = Path(LENS_COMPRESSION_WORK_ROOT) / video_path
    work_dir.mkdir(parents=True, exist_ok=True)

    src = drive_dir / "audio.wav"
    if not src.exists():
        src = drive_dir / "video.wav"

    if not src.exists():
        print(f"❌ Audio not found in: {drive_dir}")
        return 2

    local_wav = work_dir / "audio.wav"
    if not local_wav.exists():
        shutil.copy(src, local_wav)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Worker] Using device: {device}")

    print("[Worker] Loading models…")
    try:
        pipeline = Pipeline.from_pretrained(PIPELINE_ID, token=token)
        if hasattr(pipeline, "legacy"):
            pipeline.legacy = False
        pipeline.to(device)

        if hasattr(pipeline, "segmentation_batch_size"):
            pipeline.segmentation_batch_size = SEGMENTATION_BATCH_SIZE
        if hasattr(pipeline, "embedding_batch_size"):
            pipeline.embedding_batch_size = EMBEDDING_BATCH_SIZE

    except Exception as exc:
        print(f"❌ Load Error: {exc}")
        return 3

    print("[Worker] Running diarization…")
    try:
        waveform, sample_rate = torchaudio.load(str(local_wav))
        with torch.inference_mode():
            out = pipeline({"waveform": waveform, "sample_rate": sample_rate})

        if hasattr(out, "annotation"):
            diarization = out.annotation
        elif hasattr(out, "speaker_diarization"):
            diarization = out.speaker_diarization
        else:
            diarization = out

    except Exception as exc:
        print(f"❌ Diarization Error: {exc}")
        return 4

    print("[Worker] Exporting speaker embeddings (from pipeline)…")
    embeddings: dict[str, list[float]] = {}
    try:
        speaker_embeddings = getattr(out, "speaker_embeddings", None)
        if speaker_embeddings is not None:
            for i, speaker in enumerate(diarization.labels()):
                vec = speaker_embeddings[i]
                embeddings[speaker] = vec.tolist() if hasattr(vec, "tolist") else list(vec)
    except Exception as exc:
        print(f"⚠️ Embedding export warning: {exc}")

    print("[Worker] Saving…")
    try:
        dump_path = work_dir / "dump.json"
        grouped_path = work_dir / "grouped.json"

        dump_path.write_text(
            json.dumps(serialize(diarization, embeddings), indent=2), encoding="utf-8"
        )
        grouped_path.write_text(
            json.dumps(
                serialize(group(diarization, threshold=GROUP_THRESHOLD_SECONDS), embeddings),
                indent=2,
            ),
            encoding="utf-8",
        )

        shutil.copy(dump_path, drive_dir / "dump.json")
        shutil.copy(grouped_path, drive_dir / "grouped.json")

        print("✅ [Worker] Success!")
        return 0

    except Exception as exc:
        print(f"❌ Saving Error: {exc}")
        return 5


if __name__ == "__main__":
    import sys

    if len(sys.argv) < 2:
        print("Usage: worker.py <channel_id/video_id>")
        raise SystemExit(1)

    raise SystemExit(main(sys.argv[1]))
"""

Path("/content/worker.py").write_text(worker_script, encoding="utf-8")
print("✅ Wrote /content/worker.py")

✅ Wrote /content/worker.py


In [7]:
#@title 3) Scan pending videos (missing dump.json)
#@markdown Walks all folders under `LENS_COMPRESSION_YOUTUBE_ROOT` recursively to find videos without diarization output.

from pathlib import Path
from collections import Counter
import json

youtube_root = Path(LENS_COMPRESSION_YOUTUBE_ROOT)

pending_videos = []  # list of relative paths (e.g., channel/video or bucket/channel/video)

if youtube_root.exists():
    # Recursively find all .wav files
    for audio_file in youtube_root.rglob("*.wav"):
        if audio_file.name not in ("audio.wav", "video.wav"):
            continue

        video_dir = audio_file.parent

        # Skip hidden/underscore directories in the path
        if any(part.startswith("_") for part in video_dir.relative_to(youtube_root).parts):
            continue

        has_dump = (video_dir / "dump.json").exists()
        if not has_dump:
            rel_path = str(video_dir.relative_to(youtube_root))
            if rel_path not in pending_videos:
                pending_videos.append(rel_path)

# Sort for consistent processing order
pending_videos.sort()

print(f"📂 YouTube root: {youtube_root}")

# Show breakdown by parent folder (channel or bucket/channel)
group_counts = Counter(str(Path(v).parent) for v in pending_videos)
for group, count in group_counts.most_common():
    print(f"   {group}: {count} pending")

print(f"\n📝 Total pending videos: {len(pending_videos)}")
if pending_videos:
    print("First 10:")
    for v in pending_videos[:10]:
        print(f"   {v}")


📂 YouTube root: /content/drive/MyDrive/jianglens/youtube
   Interviews/UCy5tR9C0xM6tp7ha1Usq9Ag: 5 pending
   Interviews/UCsvA3BnxbPkiyHhNYhIn5cQ: 4 pending
   Interviews/UCoJhK5kMc4LjBKdiYrDtzlA: 2 pending
   Interviews/UCnXoI0GGii2VbBRIIQ2bimw: 1 pending
   Interviews/UCnYMOamNKLGVlJgRUbamveA: 1 pending
   Interviews/UCoJTOwZxbvq8Al8Qat2zgTA: 1 pending
   Interviews/UCp1HE2_FCxrAQrnMeUoCqtg: 1 pending
   Interviews/UCpR0MnZEfgrzYlo1QL8jv6A: 1 pending
   Interviews/UCrk8Y2fsR5i_5c1iTR9tZpg: 1 pending
   Interviews/UCvCym0qZPUjCWpllQMx894w: 1 pending
   Interviews/UCvQECJukTDE2i6aCoMnS-Vg: 1 pending
   Interviews/UCxEQsjgRRfGWiJJu_PDygxw: 1 pending
   Interviews/Unknown: 1 pending

📝 Total pending videos: 21
First 10:
   Interviews/UCnXoI0GGii2VbBRIIQ2bimw/dhLr7ZYdlj8
   Interviews/UCnYMOamNKLGVlJgRUbamveA/uP9Fnq23lsA
   Interviews/UCoJTOwZxbvq8Al8Qat2zgTA/t5cnf8DqJ_Q
   Interviews/UCoJhK5kMc4LjBKdiYrDtzlA/NidlGa9C0H8
   Interviews/UCoJhK5kMc4LjBKdiYrDtzlA/PvHUFfwtU3I
   Interviews/UCp

In [8]:
#@title 4) Batch processing
#@markdown Batch processing


import concurrent.futures
import os
import subprocess


def _worker_env() -> dict[str, str]:
    env = os.environ.copy()

    env["HF_TOKEN"] = HF_TOKEN
    env["HF_HOME"] = HF_HOME
    env["LENS_COMPRESSION_DRIVE_ROOT"] = LENS_COMPRESSION_DRIVE_ROOT
    env["LENS_COMPRESSION_YOUTUBE_ROOT"] = LENS_COMPRESSION_YOUTUBE_ROOT
    env["LENS_COMPRESSION_WORK_ROOT"] = LENS_COMPRESSION_WORK_ROOT

    env["DIARIZATION_PIPELINE_ID"] = DIARIZATION_PIPELINE_ID
    env["SEGMENTATION_BATCH_SIZE"] = str(SEGMENTATION_BATCH_SIZE)
    env["EMBEDDING_BATCH_SIZE"] = str(EMBEDDING_BATCH_SIZE)
    env["GROUP_THRESHOLD_SECONDS"] = str(GROUP_THRESHOLD_SECONDS)

    env["LD_LIBRARY_PATH"] = f"{LD_LIB_DIR}:{env.get('LD_LIBRARY_PATH', '')}"
    env.pop("MPLBACKEND", None)

    return env


def _run_one(video_path: str) -> None:
    """video_path is 'channel_id/video_id'."""
    print(f"⏳ [Started] {video_path}")

    cmd = [PYTHON_BIN, "/content/worker.py", video_path]
    result = subprocess.run(cmd, env=_worker_env(), capture_output=True, text=True)

    if result.returncode == 0:
        print(f"✅ [Finished] {video_path}")
        return

    print(f"❌ [Failed] {video_path}")
    print(f"--- Output for {video_path} ---\n{result.stdout}\n{result.stderr}\n----------------------------")


def process_videos(videos: list[str], *, max_workers: int) -> None:
    print(f"🚀 Starting batch processing with {max_workers} workers…")
    print(f"📅 Remaining videos: {len(videos)}")

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        list(executor.map(_run_one, videos))


if RUN_BATCH:
    process_videos(pending_videos, max_workers=MAX_WORKERS)
else:
    print("⏸️ Batch disabled. Set RUN_BATCH=True in the Config cell to start.")

🚀 Starting batch processing with 2 workers…
📅 Remaining videos: 21
⏳ [Started] Interviews/UCnXoI0GGii2VbBRIIQ2bimw/dhLr7ZYdlj8
⏳ [Started] Interviews/UCnYMOamNKLGVlJgRUbamveA/uP9Fnq23lsA
✅ [Finished] Interviews/UCnXoI0GGii2VbBRIIQ2bimw/dhLr7ZYdlj8
⏳ [Started] Interviews/UCoJTOwZxbvq8Al8Qat2zgTA/t5cnf8DqJ_Q
✅ [Finished] Interviews/UCnYMOamNKLGVlJgRUbamveA/uP9Fnq23lsA
⏳ [Started] Interviews/UCoJhK5kMc4LjBKdiYrDtzlA/NidlGa9C0H8
✅ [Finished] Interviews/UCoJhK5kMc4LjBKdiYrDtzlA/NidlGa9C0H8
⏳ [Started] Interviews/UCoJhK5kMc4LjBKdiYrDtzlA/PvHUFfwtU3I
✅ [Finished] Interviews/UCoJTOwZxbvq8Al8Qat2zgTA/t5cnf8DqJ_Q
⏳ [Started] Interviews/UCp1HE2_FCxrAQrnMeUoCqtg/nqAm_7gt_9k
✅ [Finished] Interviews/UCoJhK5kMc4LjBKdiYrDtzlA/PvHUFfwtU3I
⏳ [Started] Interviews/UCpR0MnZEfgrzYlo1QL8jv6A/DkaGDdve8SM
✅ [Finished] Interviews/UCp1HE2_FCxrAQrnMeUoCqtg/nqAm_7gt_9k
⏳ [Started] Interviews/UCrk8Y2fsR5i_5c1iTR9tZpg/3ULDVZstUjE
✅ [Finished] Interviews/UCpR0MnZEfgrzYlo1QL8jv6A/DkaGDdve8SM
⏳ [Started] Interviews/UCs

In [ ]:
#@title 5) Verify worker on a single video

import json
import os
import subprocess
from pathlib import Path


def _worker_env() -> dict[str, str]:
    env = os.environ.copy()

    env["HF_TOKEN"] = HF_TOKEN
    env["HF_HOME"] = HF_HOME
    env["LENS_COMPRESSION_DRIVE_ROOT"] = LENS_COMPRESSION_DRIVE_ROOT
    env["LENS_COMPRESSION_YOUTUBE_ROOT"] = LENS_COMPRESSION_YOUTUBE_ROOT
    env["LENS_COMPRESSION_WORK_ROOT"] = LENS_COMPRESSION_WORK_ROOT

    env["DIARIZATION_PIPELINE_ID"] = DIARIZATION_PIPELINE_ID
    env["SEGMENTATION_BATCH_SIZE"] = str(SEGMENTATION_BATCH_SIZE)
    env["EMBEDDING_BATCH_SIZE"] = str(EMBEDDING_BATCH_SIZE)
    env["GROUP_THRESHOLD_SECONDS"] = str(GROUP_THRESHOLD_SECONDS)

    env["LD_LIBRARY_PATH"] = f"{LD_LIB_DIR}:{env.get('LD_LIBRARY_PATH', '')}"
    env.pop("MPLBACKEND", None)

    return env

assert TEST_VIDEO, "Set TEST_VIDEO in Config (e.g. 'UCxxxxxxxx/s_8BKJDiwKw')"
print(f"🧪 Verifying worker on: {TEST_VIDEO}")

cmd = [PYTHON_BIN, "/content/worker.py", TEST_VIDEO]
proc = subprocess.run(cmd, env=_worker_env(), capture_output=True, text=True)
print(proc.stdout)

if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"Worker failed with exit code {proc.returncode}")

dump_path = Path(LENS_COMPRESSION_WORK_ROOT) / TEST_VIDEO / "dump.json"

if not dump_path.exists():
    raise FileNotFoundError(f"dump.json not found: {dump_path}")

data = json.loads(dump_path.read_text(encoding="utf-8"))
embeddings = data.get("embeddings", {})

print(f"📊 Embeddings found: {len(embeddings)}")
if embeddings:
    first_spk = next(iter(embeddings.keys()))
    print(f"First speaker: {first_spk}")
    print(f"Vector length: {len(embeddings[first_spk])}")

🧪 Verifying worker on: @PredictiveHistory/0HYET47Cc-E
🎬 [Worker] Processing: @PredictiveHistory/0HYET47Cc-E
[Worker] Using device: cuda
[Worker] Loading models…
[Worker] Running diarization…
[Worker] Exporting speaker embeddings (from pipeline)…
[Worker] Saving…
✅ [Worker] Success!

📊 Embeddings found: 3
First speaker: SPEAKER_00
Vector length: 256
